# Human pseudotime: choice of root

In [ ]:
import re

import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from anndata.io import read_elem
from scipy.stats import spearmanr

D = "/n/lab_storage/eddy_lab/Lab/mckinley/data"
HUMAN = f"{D}/20261107_Human_fibdec_final_nick_pdt.h5ad"      # diffusion map and pseudotime of record
STRO7 = f"{D}/20261111_Human_only_eFib_dec_final.h5ad"        # Ulrich2024_sg_Stro7_top50
SAMAP = f"{D}/20261111_samap_final.h5ad"                      # joint graph, transferred mouse pseudotime
MOUSE = f"{D}/20261110_mouse_only_eFib_dec_final_nick_pdt.h5ad"
ORDER = ["eFib1", "eFib2", "eFib3", "eFib4", "dec1", "dec2"]


## Data

In [ ]:
with h5py.File(HUMAN, "r") as f:
    obs = read_elem(f["obs"])
    dm = np.asarray(read_elem(f["obsm/X_diffmap"]))
    uns = {k: (v.decode() if isinstance(v, bytes) else v) for k, v in read_elem(f["uns/pseudotime"]).items()}
with h5py.File(STRO7, "r") as f:
    stro7_all = read_elem(f["obs"])["Ulrich2024_sg_Stro7_top50"]

on_map = np.isfinite(dm).all(axis=1)
obs, dm = obs[on_map], dm[on_map]
evals = np.array([float(x) for x in re.findall(r"[-\d.eE+]+", str(uns["diffmap_evals"]))])
sub = obs["cell_subtype"].astype(str).to_numpy()
fib = obs["cell_type"].astype(str).to_numpy() == "eFib"
secretory = obs["timepoint"].astype(str).to_numpy() == "secretory"
stro7 = stro7_all.reindex(obs.index).to_numpy(float)
print(f"{len(obs):,} nuclei; root of record {uns['root']} ({uns['root_label']}); "
      f"trajectory component DC{uns['trajectory_dc']}")


## Diffusion pseudotime from a root

In [ ]:
X = dm[:, 1:] * evals[1:] / (1 - evals[1:])


def pseudotime(root):
    distance = np.sqrt(((X - X[root]) ** 2).sum(axis=1))
    return pd.Series(distance).rank(pct=True).to_numpy()


record_root = obs.index.get_loc(uns["root"])
check = pseudotime(record_root)
print("max |recomputed - stored| x N:", np.abs(check - obs["pseudotime"].to_numpy()).max() * len(obs))


## Roots

In [ ]:
dc1 = dm[:, int(uns["trajectory_dc"])]
toward_fib = -dc1 if np.median(dc1[fib]) < np.median(dc1[~fib]) else dc1   # larger = fibroblast end


def end_of(pool):
    return int(np.argmax(np.where(pool, toward_fib, -np.inf)))


decidual_tip = int(np.argmin(toward_fib))
joint = np.minimum(pd.Series(-dm[:, 1]).rank(pct=True), pd.Series(dm[:, 4]).rank(pct=True)).to_numpy()
end10 = fib & secretory & (toward_fib >= np.quantile(toward_fib[fib & secretory], 0.9))
roots = {
    "record rule: secretory fibroblast at the DC1 end": end_of(fib & secretory),
    "lowest DC1 and highest DC4": int(np.argmax(joint)),
    "highest Stro-7 among the 10% nearest the DC1 end": int(np.argmax(np.where(end10, stro7, -np.inf))),
    "farthest from the decidual tip (no labels)": int(np.argmax(np.sqrt(((X - X[decidual_tip]) ** 2).sum(axis=1)))),
}
assert obs.index[roots["record rule: secretory fibroblast at the DC1 end"]] == uns["root"]
print("same nucleus as the record rule within eFib1:",
      roots["lowest DC1 and highest DC4"] == end_of(secretory & (sub == "eFib1")))
pd.DataFrame({"nucleus": obs.index[list(roots.values())], "subtype": sub[list(roots.values())],
              "Stro-7": stro7[list(roots.values())].round(3)}, index=list(roots))


## Diffusion map

In [ ]:
colors = dict(zip(ORDER, ["#08306b", "#2171b5", "#9ecae1", "#41ab5d", "#fdae6b", "#e6550d"]))
shown = np.random.default_rng(0).permutation(np.flatnonzero(np.isin(sub, ORDER)))
styles = [("*", "black", 300), ("X", "magenta", 140), ("P", "red", 140), ("D", "gold", 80)]
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, (i, j) in zip(axes, [(1, 2), (1, 4)]):
    ax.scatter(dm[shown, i], dm[shown, j], s=0.3, c=[colors[s] for s in sub[shown]], rasterized=True)
    for (name, r), (m, color, size) in reversed(list(zip(roots.items(), styles))):
        ax.scatter(dm[r, i], dm[r, j], s=size, marker=m, c=color, edgecolors="k", linewidths=0.5, label=name)
    ax.set_xlabel(f"DC{i}"), ax.set_ylabel(f"DC{j}"), ax.set_xticks([]), ax.set_yticks([])
for s in ORDER:
    axes[1].scatter([], [], s=20, c=colors[s], label=s)
axes[1].legend(fontsize=7, frameon=False, bbox_to_anchor=(1, 1), loc="upper left")
plt.show()


## Pseudotime by subtype

In [ ]:
pts = {name: pseudotime(r) for name, r in roots.items()}
pd.DataFrame({name: pd.Series(pt).groupby(sub).median() for name, pt in pts.items()}).reindex(ORDER).round(3)


## Twenty eFib1 roots near the DC1 end

In [ ]:
efib1_end = (sub == "eFib1") & (toward_fib >= np.quantile(toward_fib[sub == "eFib1"], 0.95))
reference = pts["lowest DC1 and highest DC4"]
rows = []
for r in np.random.default_rng(0).choice(np.flatnonzero(efib1_end), 20, replace=False):
    pt = pseudotime(r)
    medians = pd.Series(pt).groupby(sub).median().reindex(ORDER)
    rows.append({"nucleus": obs.index[r], "Spearman with eFib1-rule pseudotime": spearmanr(pt, reference).statistic,
                 "order": " < ".join(medians.sort_values().index)})
pd.DataFrame(rows).round(3)


## Agreement with the transferred mouse pseudotime

In [ ]:
with h5py.File(SAMAP, "r") as f:
    joint = read_elem(f["obs"])
    W = read_elem(f["obsp/connectivities"]).tocsr()
mouse_pt = read_elem(h5py.File(MOUSE, "r")["obs"])["pseudotime"]
strip = lambda names, sp: pd.Index([n[len(sp) + 1:] if n.startswith(sp + "_") else n for n in names.astype(str)])
species = joint["species"].astype(str).to_numpy()
hs, mm = np.flatnonzero(species == "hs"), np.flatnonzero(species == "mm")
m_pt = mouse_pt.reindex(strip(joint.index[mm], "mm")).to_numpy(float)
m_type = joint["cell_type"].astype(str).to_numpy()[mm]
has_pt = np.isfinite(m_pt)
rows_in_joint = strip(joint.index[hs], "hs").get_indexer(obs.index)
B = W[hs][:, mm][:, has_pt].tocsr()[rows_in_joint]
weight = np.asarray(B.sum(axis=1)).ravel()


def transfer(values):
    with np.errstate(invalid="ignore", divide="ignore"):
        return np.asarray(B @ values).ravel() / weight


transferred = transfer(m_pt[has_pt])
rng = np.random.default_rng(0)
shuffles = []
for _ in range(100):
    shuffled = m_pt[has_pt].copy()
    for t in ("endometrial_fibroblast", "decidual_cell"):
        idx = np.flatnonzero(m_type[has_pt] == t)
        shuffled[idx] = rng.permutation(shuffled[idx])
    shuffles.append(transfer(shuffled))

rows = {}
for name, pt in pts.items():
    row = {}
    for group, mask in (("all", np.ones(len(pt), bool)), ("fibroblasts", fib), ("decidual", ~fib)):
        ok = mask & np.isfinite(transferred)
        null = [spearmanr(pt[ok], s[ok]).statistic for s in shuffles]
        row[f"rho {group}"] = spearmanr(pt[ok], transferred[ok]).statistic
        row[f"shuffle max {group}"] = max(null)
    rows[name] = row
pd.DataFrame(rows).T.round(3)


## Pseudotime in tissue

In [ ]:
shown = ["record rule: secretory fibroblast at the DC1 end", "lowest DC1 and highest DC4"]
xy = obs[["x_um_dbscan", "y_um_dbscan"]].to_numpy(float)
fig, axes = plt.subplots(2, 2, figsize=(12, 11))
for row, section in zip(axes, ["secretory", "menstrual"]):
    keep = fib & (obs["timepoint"].astype(str).to_numpy() == section) & np.isfinite(xy).all(axis=1)
    for ax, name in zip(row, shown):
        im = ax.scatter(xy[keep, 0], xy[keep, 1], s=1, c=pts[name][keep], cmap="viridis", vmin=0, vmax=1)
        ax.set_title(f"{section}: {name}", fontsize=9), ax.set_aspect("equal"), ax.axis("off")
fig.colorbar(im, ax=axes, shrink=0.5, label="pseudotime (fibroblasts)")
plt.show()
